In [ ]:
import sys
import json
import copy
import time
from pathlib import Path
from typing import Any, Dict, Optional, List

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.config import DISTANCE_CLASS_FACTORS

class SidecarManager:
    """
    Istanziare una singola volta per filepath (.json).
    La stessa accortezza va realizzata non solo nello stesso notebook,
    ma anche tra notebook DIVERSI.
    Evitare cioè modifiche concorrenti, visto che non c'è nessun lock che le impedisca.
    Nello stesso notebook c'è un WARNING, ma tra notebook diversi non c'è avviso.
    """

    # Tiene traccia dei path per cui è stata creata un istanza.
    #  L'utilizzo di una cache infatti rende problematica la presenza di più istanze:
    #  ciascuna avrebbe una propria cache in RAM indipendente con il rischio che il loro contenuto
    #  diverga senza avvisi.
    #  Un implementazione singleton avrebbe complicato molto il codice, invece tramite
    #  questa accortezza se si istanzia una seconda istanza viene stampato un avviso (non bloccante)
    _seen_paths: set = set()

    DEFAULT_PALETTE = [
        "#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f",
        "#edc949", "#af7aa1", "#ff9da7", "#9c755f", "#bab0ab"
    ] # Colori di Default se non ne vengono assegnati altri

    def __init__(self, filepath="sidecar_edits.json"):
        # Inizializzazione di default; modificabile passandogli un diverso path come parametro
        self.filepath = (
            Path(filepath).expanduser().resolve()
            if filepath else Path("sidecar_edits.json").resolve()
        )

        # Verifica istanziazione singola
        if self.filepath in SidecarManager._seen_paths:
            print(
                f"[WARNING] Creata una seconda istanza di SidecarManager per lo stesso file "
                f"({self.filepath}). Ogni istanza mantiene una propria cache in RAM: se non e' "
                f"la stessa istanza ad essere condivisa e riutilizzata ovunque, le due cache "
                f"possono disallinearsi silenziosamente. Crea una sola istanza e passala "
                f"esplicitamente a tutti i componenti che leggono/scrivono il sidecar."
            )
        SidecarManager._seen_paths.add(self.filepath)

        # Crea la cache
        self._cache: Optional[Dict[str, Any]] = None

        self._ensure_file_exists()

    ### Accesso e gestione dati e cache

    def _ensure_file_exists(self):
        self.filepath.parent.mkdir(parents=True, exist_ok=True)
        if not self.filepath.exists():
            self.reset_all()

    @property
    def data(self) -> dict:
        """Accesso ai dati aggiornati forzando automaticamente il reload"""
        return self.load_data(force_reload=False)

    def load_data(self, force_reload: bool=False) -> dict:
        """
        Carica i dati del sidecaar.
        Sfrutta la cache in RAM per evitare I/O ripetuto e eventuale collo di bottiglia.
        Restituisce sempre la deepcopy, così eventuali mutazioni da parte del chiamante
        (prima di un save_data) non sporcano la cache_interna.
        """
        if self._cache is not None and not force_reload:
            return copy.deepcopy(self._cache)

        if self.filepath.exists():
            try:
                with open(self.filepath, "r", encoding="utf-8") as f:
                    self._cache = json.load(f)
                return copy.deepcopy(self._cache)
            except Exception as e:
                # Il file esiste ma non è leggibile (es. JSON corrotto per
                # un'interruzione a metà scrittura). Prima di scartarlo ne
                # salviamo una copia: la prossima save_data() sovrascriverà
                # il file originale, quindi senza backup il contenuto
                # eventualmente ancora recuperabile andrebbe perso per sempre.
                backup_path = self.filepath.with_suffix(f".corrotto.{int(time.time())}.json")
                try:
                    self.filepath.replace(backup_path)
                    print(f"<[WARNING]> File sidecar illeggibile ({e}). "
                          f"Backup del file originale salvato in: {backup_path}")
                except Exception:
                    print(f"<[WARNING]> File sidecar illeggibile ({e}) e impossibile crearne un backup.")


        default_structure = {
            "modified_adjacencies": {},
            "tag_overrides": {},
            "global_tags": {},
            "predetermined_queries": {}
        }
        self._cache = default_structure
        return copy.deepcopy(self._cache)

    def save_data(self, data: dict):
        """Salva il dizionario su disco in modo atomico e aggiorna la cache in RAM."""
        self.filepath.parent.mkdir(parents=True, exist_ok=True)

        # Scrittura atomica: si scrive prima su un file temporaneo e poi lo
        # si sostituisce al file definitivo. Path.replace (os.replace) e'
        # atomico su POSIX, quindi anche un'interruzione a meta' (crash,
        # restart del kernel) non puo' lasciare sidecar_edits.json in uno
        # stato troncato/corrotto: resta o la vecchia versione, o gia'
        # completa quella nuova.
        tmp_path = self.filepath.with_suffix(self.filepath.suffix + ".tmp")
        with open(tmp_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        tmp_path.replace(self.filepath)

        self._cache = copy.deepcopy(data)

    def invalidate_cache(self):
        """Forza la rilettura da filesystem alla successiva chiamata di load_data()."""
        self._cache = None

    def release_path(self) -> None:
        """
        Rimuove il path di questa istanza dal registro delle istanze viste.
        Da chiamare quando l'istanza non verrà più usata (es. in PreTagger.close()),
        così una nuova SidecarManager sullo stesso file non genera un falso avviso
        di "seconda istanza". NON funziona tra notebook diversi.
        """
        SidecarManager._seen_paths.discard(self.filepath)

    def get_global_tags(self) -> dict:
        """
        Restituisce il dizionario dei tag attivi, con contatore e colore corrente.
        """
        data = self.load_data()
        return data.get("global_tags", {})

    ### Manipolazione Classi di Distanza
    def save_pairwise_class_edits(self, chunk_id_1: str, chunk_id_2: str, distance_class: Optional[str]):
        """
        Salva o aggiorna la classe di distanza tra una coppia di chunk.
        Garantisce simmetria (chunk_1 <-> chunk_2) e pulizia automatica se la classe
        dovesse essere "INVARIATI" o non presente in DISTANCE_CLASS_FACTORS.
        Reindirizza automaticamente a save_predetermined_query_adjacency se uno degli ID riguarda una query pre-determinata
        """
        c1, c2 = str(chunk_id_1), str(chunk_id_2)
        if c1 == c2:
            return

        # Instradamento automatico verso le query pre-determinate
        if c1.startswith("predq_") or c2.startswith("predq_"):
            self.save_predetermined_query_adjacency(c1, c2, distance_class)
            return

        data = self.load_data()
        adj = data.setdefault("modified_adjacencies", {})

        norm_class = str(distance_class).upper().strip() if distance_class is not None else "INVARIATI"
        if norm_class not in DISTANCE_CLASS_FACTORS:
            norm_class = "INVARIATI"

        if norm_class == "INVARIATI":
            if c1 in adj and c2 in adj[c1]:
                del adj[c1][c2]
                if not adj[c1]:
                    del adj[c1]

            if c2 in adj and c1 in adj[c2]:
                del adj[c2][c1]
                if not adj[c2]:
                    del adj[c2]

            print(f"<<| Adiacenza rimossa (ripristino a INVARIATI) tra [{c1}] e [{c2}] |>>")
        else:
            adj.setdefault(c1, {})[c2] = {"distance_class": norm_class}
            adj.setdefault(c2, {})[c1] = {"distance_class": norm_class}
            print(f"<<| Modifica adiacenza salvata tra [{c1}] e [{c2}]: classe={norm_class} |>>")

        self.save_data(data)

    def save_pairwise_class_edits_batch(self, edits: List[Dict[str, Any]]):
        """
        Salva un blocco di modifiche adiacenze in un'unica operazione di I/O.
        Ogni voce di 'edits' deve contenere: 'chunk_1', 'chunk_2', 'distance_class'.
        Smista automaticamente gli elementi coinvolgenti 'predq_' al relativo batch dedicato.
        """
        if not edits or not isinstance(edits, list):
            return

        standard_edits = []
        predq_edits = []

        for edit in edits:
            if edit and "chunk_1" in edit and "chunk_2" in edit:
                c1, c2 = str(edit["chunk_1"]), str(edit["chunk_2"])
                if c1.startswith("predq_") or c2.startswith("predq_"):
                    predq_edits.append(edit)
                else:
                    standard_edits.append(edit)

        # Esegue prima l'eventuale sotto-batch per le query pre-determinate
        if predq_edits:
            self.save_predetermined_query_adjacencies_batch(predq_edits)

        if not standard_edits:
            return

        data = self.load_data()
        adj = data.setdefault("modified_adjacencies", {})
        updated = False

        for edit in standard_edits:
            c1, c2 = str(edit["chunk_1"]), str(edit["chunk_2"])
            if c1 == c2:
                continue
            d_cls = edit.get("distance_class")
            norm_class = str(d_cls).upper().strip() if d_cls is not None else "INVARIATI"

            if norm_class not in DISTANCE_CLASS_FACTORS:
                norm_class = "INVARIATI"

            if norm_class == "INVARIATI":
                if c1 in adj and c2 in adj[c1]:
                    del adj[c1][c2]
                    if not adj[c1]:
                        del adj[c1]
                if c2 in adj and c1 in adj[c2]:
                    del adj[c2][c1]
                    if not adj[c2]:
                        del adj[c2]
            else:
                adj.setdefault(c1, {})[c2] = {"distance_class": norm_class}
                adj.setdefault(c2, {})[c1] = {"distance_class": norm_class}
            updated = True

        if updated:
            self.save_data(data)
            print(f"<<| Batch adiacenze standard salvato: {len(standard_edits)} elementi processati |>>")
    # Estrazione adiacenze
    def get_chunk_adjacencies(self, chunk_id: str) -> Dict[str, Dict[str, str]]:
        """
        Restituisce il dizionario delle adiacenze modificate per un singolo chunk.
        es. return format: {'chunk_2': {'distance_class': 'AVVICINATI'}}
        """
        data = self.load_data()
        adj = data.get("modified_adjacencies", {})
        return adj.get(str(chunk_id), {})

    def get_all_modified_adjacencies(self) -> Dict[str, Dict[str, Dict[str, str]]]:
        """Restituisce l'intero dizionario delle adiacenze modificate."""
        data = self.load_data()
        return data.get("modified_adjacencies", {})

    ### Manipolazione TAG
    def add_tag_override(self, chunk_id: str, tag: str, color: Optional[str] = None):
        """
        Aggiunge un singolo tag a un chunk o a una query pre-determinata.
        Aggiorna global_tags SOLO per i chunk standard del database.
        """
        if not tag:
            return
        self.add_tag_override_for_chunk(chunk_id, [tag])

    def add_tag_override_for_chunk(self, chunk_id: str, tags: List[str]):
        """
        Assegna una lista di tag a un singolo chunk o a una query pre-determinata, aggiornando il registro globale
        ed eseguendo un uica operazione di scrittura su disco per l'intero chunk.
        Da utilizzare ad esempio per il pre-tagging, ottimizzando così l'operazione.
        Usato anche nel singolo tag_override per centralizzare l'operazione.
        Aggiorna global_tags SOLO per i chunk del database.
        """
        if not tags:
            return

        data = self.load_data()
        chunk_id = str(chunk_id)

        # Gestione Query Pre-Determinata (NON tocca global_tags)
        #  I tag graficamente aggiunti, sono considerati assigned_tags; conv_tags e auto_tags non possono essere manipolati.
        if chunk_id.startswith("predq_"):
            pred_queries = data.setdefault("predetermined_queries", {})
            query_entry = pred_queries.setdefault(chunk_id, {})
            current_tags = query_entry.setdefault("graphically_assigned_tags", [])

            updated = False
            for t in tags:
                if t not in current_tags:
                    current_tags.append(t)
                    updated = True

            if updated:
                self.save_data(data)
            return

        # Gestione Chunk Standard del Database
        if "global_tags" not in data:
            data["global_tags"] = {}
        if "tag_overrides" not in data:
            data["tag_overrides"] = {}

        if chunk_id not in data["tag_overrides"]:
            data["tag_overrides"][chunk_id] = {"user_tags": []}

        current_tags = data["tag_overrides"][chunk_id].setdefault("user_tags", [])

        updated = False
        for t in tags:
            if t not in current_tags:
                current_tags.append(t)
                updated = True

                if t not in data["global_tags"]:
                    used_colors = {
                        t_info.get("color") for t_info in data["global_tags"].values()
                        if isinstance(t_info, dict)
                    }
                    available = [c for c in self.DEFAULT_PALETTE if c not in used_colors]
                    col = (
                        available[0] if available
                        else self.DEFAULT_PALETTE[len(data["global_tags"]) % len(self.DEFAULT_PALETTE)]
                    )
                    data["global_tags"][t] = {"count": 1, "color": col}
                else:
                    data["global_tags"][t]["count"] += 1

        if updated:
            self.save_data(data)

    def add_tag_overrides_batch(self, chunk_tags_map: Dict[str, List[str]]) -> None:
        """
        Assegna tag a più chunk in un'unica operazione di lettura+scrittura su disco.
        chunk_tags_map: {chunk_id: [tag1, tag2, ...], ...}
        """
        if not chunk_tags_map:
            return

        data = self.load_data()
        if "tag_overrides" not in data:
            data["tag_overrides"] = {}
        if "global_tags" not in data:
            data["global_tags"] = {}

        updated = False

        for chunk_id, tags in chunk_tags_map.items():
            if not tags:
                continue

            cid_str = str(chunk_id)

            # Ramo Query Pre-Determinata
            if cid_str.startswith("predq_"):
                pred_queries = data.setdefault("predetermined_queries", {})
                query_entry = pred_queries.setdefault(cid_str, {})
                current_tags = query_entry.setdefault("graphically_assigned_tags", [])

                for t in tags:
                    if t not in current_tags:
                        current_tags.append(t)
                        updated = True
                continue

            # Ramo Chunk Standard
            if cid_str not in data["tag_overrides"]:
                data["tag_overrides"][cid_str] = {"user_tags": []}

            current_tags = data["tag_overrides"][cid_str].setdefault("user_tags", [])

            for t in tags:
                if t not in current_tags:
                    current_tags.append(t)
                    updated = True

                    if t not in data["global_tags"]:
                        used_colors = {
                            t_info.get("color") for t_info in data["global_tags"].values()
                            if isinstance(t_info, dict)
                        }
                        available = [c for c in self.DEFAULT_PALETTE if c not in used_colors]
                        col = (
                            available[0] if available
                            else self.DEFAULT_PALETTE[len(data["global_tags"]) % len(self.DEFAULT_PALETTE)]
                        )
                        data["global_tags"][t] = {"count": 1, "color": col}
                    else:
                        data["global_tags"][t]["count"] += 1

        if updated:
            self.save_data(data)

    def remove_tag_override(self, chunk_id: str, tag: str):
        """
        Rimuove un tag da un chunk o da una query pre-determinata, decrementando il contatore globale.
        Se questo scende a 0, rimuove il tag da tale registro globale.
        Decrementa global_tags SOLO per i chunk del database.
        """
        data = self.load_data()
        chunk_id = str(chunk_id)

        # Rimozione da Query Pre-Determinata
        if chunk_id.startswith("predq_"):
            pred_queries = data.get("predetermined_queries", {})
            if chunk_id in pred_queries:
                current_tags = pred_queries[chunk_id].get("graphically_assigned_tags", [])
                if tag in current_tags:
                    current_tags.remove(tag)
                    self.save_data(data)
            return

        # Rimozione da Chunk Standard
        tag_removed = False
        if "tag_overrides" in data and chunk_id in data["tag_overrides"]:
            current_tags = data["tag_overrides"][chunk_id].get("user_tags", [])
            if tag in current_tags:
                current_tags.remove(tag)
                tag_removed = True
                if not current_tags:
                    del data["tag_overrides"][chunk_id]

        if tag_removed and "global_tags" in data and tag in data["global_tags"]:
            data["global_tags"][tag]["count"] -= 1
            if data["global_tags"][tag]["count"] <= 0:
                del data["global_tags"][tag]
            self.save_data(data)

    ### RESET file sidecar
    def reset_all(self):
        """Ripristina il file sidecar azzerando le modifiche (UNDO globale)."""
        self.save_data({
            "modified_adjacencies": {},
            "tag_overrides": {},
            "global_tags": {},
            "predetermined_queries": {}
        })
        print("<<! File sidecar ripristinato allo stato iniziale !>>")

    ### Gestione Query Pre-Determinate
    # Estrazione dati
    def get_predetermined_queries(self) -> Dict[str, Dict[str, Any]]:
        """Restituisce l'intero dizionario delle query pre-determinate salvate."""
        data = self.load_data()
        return data.get("predetermined_queries", {})

    def get_predetermined_query(self, pred_query_id: str) -> Dict[str, Any]:
        """Restituisce i dati e le adiacenze per una specifica query pre-determinata."""
        queries = self.get_predetermined_queries() or {}
        return queries.get(str(pred_query_id), {})

    def get_predetermined_query_adjacencies(self, pred_query_id: str) -> Dict[str, Dict[str, str]]:
        """
        Restituisce il dizionario delle adiacenze modificate per una specifica query pre-determinata.
        Format di ritorno:
        {'chunk_12': {'distance_class': 'AVVICINATI'}}
        """
        query_entry = self.get_predetermined_query(pred_query_id)
        return query_entry.get("adjacencies", {})

    # Salvataggio adiacenze
    def save_predetermined_query_adjacency(self, chunk_id_1: str, chunk_id_2: str, distance_class: Optional[str]):
        """
        Salva o aggiorna l'adiacenza tra una query pre-determinata (ID con prefisso 'predq_') e un chunk.
        Se la classe e' INVARIATI, rimuove la voce dal JSON.
        """
        c1, c2 = str(chunk_id_1), str(chunk_id_2)
        if c1 == c2:
            return

        if c1.startswith("predq_"):
            pred_id, target_chunk = c1, c2
        elif c2.startswith("predq_"):
            pred_id, target_chunk = c2, c1
        else:
            self.save_pairwise_class_edits(c1, c2, distance_class)
            return

        data = self.load_data()
        pred_queries = data.setdefault("predetermined_queries", {})
        query_entry = pred_queries.setdefault(pred_id, {"adjacencies": {}, "graphically_assigned_tags": []})
        adj = query_entry.setdefault("adjacencies", {})

        norm_class = str(distance_class).upper().strip() if distance_class is not None else "INVARIATI"
        if norm_class not in DISTANCE_CLASS_FACTORS:
            norm_class = "INVARIATI"

        if norm_class == "INVARIATI":
            if target_chunk in adj:
                del adj[target_chunk]
            print(f"<<| Adiacenza query pre-determinata rimossa (INVARIATI) tra [{pred_id}] e [{target_chunk}] |>>")
        else:
            adj[target_chunk] = {"distance_class": norm_class}
            print(f"<<| Modifica adiacenza query pre-determinata salvata tra [{pred_id}] e [{target_chunk}]: classe={norm_class} |>>")

        self.save_data(data)

    def save_predetermined_query_adjacencies_batch(self, edits: List[Dict[str, Any]]):
        """Salva un blocco di modifiche di adiacenza per query pre-determinate in un'unica operazione di I/O."""
        if not edits or not isinstance(edits, list):
            return

        data = self.load_data()
        pred_queries = data.setdefault("predetermined_queries", {})
        updated = False

        for edit in edits:
            if not edit or "chunk_1" not in edit or "chunk_2" not in edit:
                continue
            c1, c2 = str(edit["chunk_1"]), str(edit["chunk_2"])
            if c1 == c2:
                continue

            if c1.startswith("predq_"):
                pred_id, target_chunk = c1, c2
            elif c2.startswith("predq_"):
                pred_id, target_chunk = c2, c1
            else:
                continue

            query_entry = pred_queries.setdefault(pred_id, {"adjacencies": {}, "graphically_assigned_tags": []})
            adj = query_entry.setdefault("adjacencies", {})

            d_cls = edit.get("distance_class")
            norm_class = str(d_cls).upper().strip() if d_cls is not None else "INVARIATI"
            if norm_class not in DISTANCE_CLASS_FACTORS:
                norm_class = "INVARIATI"

            if norm_class == "INVARIATI":
                if target_chunk in adj:
                    del adj[target_chunk]
            else:
                adj[target_chunk] = {"distance_class": norm_class}
            updated = True

        if updated:
            self.save_data(data)
            print(f"<<| Batch adiacenze query pre-determinate salvato: {len(edits)} elementi processati |>>")


In [ ]:
import sys
import pathlib
from pathlib import Path
import anywidget
import traitlets
project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import BASE_GRAPH_DISTANCE, DISTANCE_CLASS_FACTORS
from src.sidecar_manager import SidecarManager

class ChunkGraphWidget(anywidget.AnyWidget):
    _esm = pathlib.Path(__file__).parent / "frontend" / "chunk_graph.js"

    # Traitlets per stato del grafo
    graph_data = traitlets.Dict({"nodes": [], "links": []}).tag(sync=True)
    selected_tag = traitlets.Dict({}).tag(sync=True)
    global_tags = traitlets.Dict({}).tag(sync=True)
    tag_action = traitlets.Dict({}).tag(sync=True)

    #Traitlets per modifiche sulle classi di distanza in batch -- assicura di non perdere edit
    pairwise_class_edit = traitlets.Dict({}).tag(sync=True)
    pairwise_edits_batch = traitlets.List([]).tag(sync=True)

    def __init__(self, sidecar=None, sidecar_path=None, **kwargs):
        """
        Per evitare di avere una doppia cache del sidecar_manager, è bene passare il sidecar_mngr
        e non il sidecar_path per evitare disallineamenti della cache RAM. L'opzione è stata lasciata per test e legacy, ma farlo rischia
        di dare un WARNING che è fondamentale non ignorare se non con coscienza.
        """
        super().__init__(**kwargs)

        if isinstance(sidecar, SidecarManager):
            self.sidecar = sidecar
        elif sidecar_path:
            self.sidecar = SidecarManager(filepath=sidecar_path)
        else:
            self.sidecar = SidecarManager()

        self._raw_graph_data = {"nodes": [], "links": []}

        self.observe(self._on_pairwise_edit, names=["pairwise_class_edit"])
        self.observe(self._on_pairwise_edits_batch, names=["pairwise_edits_batch"])
        self.observe(self._on_tag_action, names=["tag_action"])
        self.refresh_global_tags()

    ### Caricamento grafo
    def load_graph(self, raw_graph_data):
        """Carica il grafo applicando immediatamente i distance_factor salvati nel sidecar."""
        self._raw_graph_data = raw_graph_data
        data = self.sidecar.load_data()
        adj = data.get("modified_adjacencies", {})
        overrides = data.get("tag_overrides", {})

        # Info sui nodi
        nodes = raw_graph_data.get("nodes", [])
        enriched_nodes = []
        valid_node_ids = set()

        for node in nodes:
            n_copy = dict(node)
            cid = str(n_copy.get("id"))
            valid_node_ids.add(cid)
            if cid in overrides:
                n_copy["user_tags"] = overrides[cid].get("user_tags", [])
            elif "user_tags" not in n_copy:
                n_copy["user_tags"] = n_copy.get("tags", [])
            enriched_nodes.append(n_copy)

        # Archi e struttura a adiacenza
        raw_links = raw_graph_data.get("links", [])
        enriched_links = []
        existing_pairs = set()

        for link in raw_links:
            l_copy = dict(link)
            src = str(l_copy["source"]["id"] if isinstance(l_copy["source"], dict) else l_copy["source"])
            tgt = str(l_copy["target"]["id"] if isinstance(l_copy["target"], dict) else l_copy["target"])

            pair_key = tuple(sorted([src, tgt]))
            existing_pairs.add(pair_key)

            distance_class = "INVARIATI"
            if src in adj and tgt in adj[src]:
                distance_class = adj[src][tgt].get("distance_class", "INVARIATI")
            elif tgt in adj and src in adj[tgt]:
                distance_class = adj[tgt][src].get("distance_class", "INVARIATI")

            factor = DISTANCE_CLASS_FACTORS.get(distance_class, 1.0)

            l_copy["source"] = src
            l_copy["target"] = tgt
            l_copy["distance_class"] = distance_class
            l_copy["distance_factor"] = factor
            enriched_links.append(l_copy)

        # Aggiunge archi del grafo non presenti nel kg originale
        #  Ma derivanti da alterazioni del sidecar
        for src, targets in adj.items():
            for tgt, info in targets.items():
                pair_key = tuple(sorted([str(src), str(tgt)]))
                if pair_key not in existing_pairs:
                    d_cls = info.get("distance_class", "INVARIATI")
                    if d_cls != "INVARIATI":
                        existing_pairs.add(pair_key)
                        factor = DISTANCE_CLASS_FACTORS.get(d_cls, 1.0)
                        enriched_links.append({
                            "source": str(src),
                            "target": str(tgt),
                            "distance_class": d_cls,
                            "distance_factor": factor
                        })

       # Mantiene soli gli archi con sia source che target trai nodi attivi
        valid_links = [
            link for link in enriched_links
            if str(link["source"]) in valid_node_ids and str(link["target"]) in valid_node_ids
        ]

        self.graph_data = {
            "nodes": enriched_nodes,
            "links": valid_links,
            "base_distance": BASE_GRAPH_DISTANCE
        }
        self.refresh_global_tags()

    ### Gestione TAG
    def refresh_global_tags(self):
        """Sincronizza il dizionatio dei tag globali con il frontend"""
        self.global_tags = self.sidecar.get_global_tags()

    def _on_tag_action(self, change):
        """Gestisce le azioni di aggiunta e rimozione dei TAG inviate da JavaScript"""
        action_data = change["new"]
        if not action_data:
            return

        action = action_data.get("action")
        chunk_id = action_data.get("chunk_id")
        tag = action_data.get("tag")
        color = action_data.get("color")

        if action == "add" and chunk_id and tag:
            self.sidecar.add_tag_override(chunk_id, tag, color=color)
        elif action == "remove" and chunk_id and tag:
            self.sidecar.remove_tag_override(chunk_id, tag)

        self.refresh_global_tags()
        self._update_node_tags_in_graph_data(chunk_id)

    def _update_node_tags_in_graph_data(self, chunk_id):
        """
        Aggiorna i tag del nodo specifico in graph_data per forzare il ridisegno.
        Semplicemente ricrea e riposiziona i dati (uguali) nel nodo del chunk per forzare il re-render
        """
        data = self.sidecar.load_data()
        chunk_overrides = data.get("tag_overrides", {}).get(str(chunk_id), {})
        user_tags = chunk_overrides.get("user_tags", [])

        new_graph_data = dict(self.graph_data)
        nodes = [dict(n) for n in new_graph_data.get("nodes", [])]
        for n in nodes:
            if str(n.get("id")) == str(chunk_id):
                n["user_tags"] = user_tags
        new_graph_data["nodes"] = nodes
        self.graph_data = new_graph_data

    ### Gestione Classi di Distanza
    def _on_pairwise_edit(self, change):
        """Gestisce una singola modifica di distanza inviata da JS -- versione meno sicura di pairwise_edits_batch"""
        edit = change["new"]
        if edit and "chunk_1" in edit and "chunk_2" in edit:
            c1 = edit["chunk_1"]
            c2 = edit["chunk_2"]
            d_cls = edit.get("distance_class")
            self.sidecar.save_pairwise_class_edits(c1, c2, d_cls)
            if self._raw_graph_data.get("nodes"):
                self.load_graph(self._raw_graph_data)

    def _on_pairwise_edits_batch(self, change):
        """
        Gestisce un blocco (tramite lista) di modifiche simultanee inviato in una sola volta.
        Si verifica quando il trascinamento di 1 nodo altera la distanza con più nodi ad esso collegati.
        Delegata al SidecarManager.
        """
        edits = change["new"]
        if edits and isinstance(edits, list):
            self.sidecar.save_pairwise_class_edits_batch(edits)
            if self._raw_graph_data.get("nodes"):
                self.load_graph(self._raw_graph_data)


In [ ]:
import sys
from pathlib import Path
import networkx as nx
import numpy as np
project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.config import SIMILARITY_THRESHOLD

class KnowledgeGraphBuilder:
    def __init__(self):
        self.graph = nx.Graph()
        self._vectors = {}

    def add_chunk_node(self, chunk_id: str, text: str, vector: list = None, tags: list = None):
        """Aggiunge un nodo al grafo e ne memorizza opzionalmente il vettore di embedding."""
        self.graph.add_node(chunk_id, text=text, tags=tags or [])
        if vector is not None:
            self._vectors[chunk_id] = np.array(vector, dtype=float)

    def add_relation(self, source_id: str, target_id: str, weight: float = 1.0):
        """Aggiunge un arco tra due nodi con un peso=similarità."""
        self.graph.add_edge(source_id, target_id, weight=weight)

    def auto_connect_nodes(self, records: list=None):
        """
        Calcola la Cosine Similarity tra tutte le coppie di nodi e crea un arco se supera SIMILARITY_THRESHOLD.

        - Se 'records' è None (PREDILETTO): usa self._vectors memorizzati internamente.
        - Se 'records' è fornito (LEGACY): usa la lista esterna di oggetti con attributi .id e .vector.
        """
        def cosine_sim(v1, v2):
            norm1 = np.linalg.norm(v1)
            norm2 = np.linalg.norm(v2)
            if norm1 == 0 or norm2 == 0:
                return 0.0
            return float(np.dot(v1, v2) / (norm1 * norm2))

        # Modalità LEGACY con lista esterna
        if records is not None:
            n = len(records)
            for i in range(n):
                for j in range(i + 1, n):
                    v1, v2 = records[i].vector, records[j].vector
                    sim = float(cosine_sim(v1, v2))
                    if sim >= SIMILARITY_THRESHOLD:
                        self.add_relation(
                            source_id=str(records[i].id),
                            target_id=str(records[j].id),
                            weight=sim
                        )
        else: # Modalità PREDILETTA con dizionario interno
            items = list(self._vectors.items())
            n = len(items)
            for i in range(n):
                id1, v1 = items[i]
                for j in range(i + 1, n):
                    id2, v2 = items[j]
                    sim = float(cosine_sim(v1, v2))
                    if sim >= SIMILARITY_THRESHOLD:
                        self.add_relation(
                            source_id=id1,
                            target_id=id2,
                            weight=sim
                        )

    def to_json_data(self) -> dict:
        data = nx.node_link_data(self.graph)
        if "edges" in data and "links" not in data:
            data["links"] = data.pop("edges")
        return data


In [ ]:
import * as d3 from "https://esm.sh/d3@7";

export function render({ model, el }) {
  el.innerHTML = "";

  function str(val) {
    return String(val);
  }

  function getEdgeKey(src, tgt) {
    const s = str(src);
    const t = str(tgt);
    return s < t ? `${s}_AND_${t}` : `${t}_AND_${s}`;
  }

  let currentSimulation = null;
  let renderPending = false;
  let selectedNodeId = null;
  let selectedEdgeKey = null; // Memorizza la chiave dell'arco selezionato
  let currentNodes = [];

  /*
  //palette blu-azzurro-grigi-arancione-rosso
  const EDGE_CLASSES = [
    { name: "MOLTO_AVVICINATI", label: "  Molto Avvicinati", color: "#1d4ed8" },
    { name: "AVVICINATI", label: "  Avvicinati", color: "#06b6d4" },
    { name: "INVARIATI", label: "  Invariati", color: "#64748b" },
    { name: "ALLONTANATI", label: "  Allontanati", color: "#f97316" },
    { name: "MOLTO_ALLONTANATI",label: "  Molto Allontanati",color: "#dc2626" }
  ];
  */

  /*
  // palette alternativa verde scuro-verde chiaro-grigio-arancione-rosso
  const EDGE_CLASSES = [
      { name: "MOLTO_AVVICINATI", label: "  Molto Avvicinati", color: "#15803d" },
      { name: "AVVICINATI", label: "  Avvicinati", color: "#22c55e" },
      { name: "INVARIATI", label: "  Invariati", color: "#64748b" },
      { name: "ALLONTANATI", label: "  Allontanati", color: "#f97316" },
      { name: "MOLTO_ALLONTANATI", label: "  Molto Allontanati", color: "#ef4444" }
  ];
  */

  // palette alternativa viola-azzurro-grigio-arancione-rosso
  const EDGE_CLASSES = [
      { name: "MOLTO_AVVICINATI", label: "  Molto Avvicinati", color: "#e314b1" },
      { name: "AVVICINATI", label: "  Avvicinati", color: "#06b6d4" },
      { name: "INVARIATI", label: "  Invariati", color: "#64748b" },
      { name: "ALLONTANATI", label: "  Allontanati", color: "#f97316" },
      { name: "MOLTO_ALLONTANATI", label: "  Molto Allontanati", color: "#ef4444" }
  ];

  const EDGE_CLASS_COLORS = Object.fromEntries(
    EDGE_CLASSES.map(c => [c.name, c.color])
  );

  // --- LAYOUT PRINCIPALE FLEXBOX ---
  const mainContainer = d3.select(el)
    .append("div")
    .style("display", "flex")
    .style("flex-direction", "row")
    .style("gap", "15px")
    .style("font-family", "system-ui, -apple-system, sans-serif")
    .style("color", "#1e293b");

  // --- COLONNA SINISTRA: GRAFO D3.js ---
  const graphContainer = mainContainer.append("div")
    .style("position", "relative")
    .style("width", "580px")
    .style("height", "420px");

  const width = 580;
  const height = 420;

  const svg = graphContainer.append("svg")
    .attr("width", width)
    .attr("height", height)
    .style("background", "#f8fafc")
    .style("border", "1px solid #cbd5e1")
    .style("border-radius", "8px")
    .style("cursor", "grab");

  const defs = svg.append("defs");
  const g = svg.append("g");

  const zoom = d3.zoom()
    .scaleExtent([0.2, 4])
    .on("zoom", (event) => {
      g.attr("transform", event.transform);
    });

  svg.call(zoom);

  // --- DESELEZIONE AL CLICK SULLO SFONDO SVG ---
  svg.on("click", (event) => {
    if (event.target === svg.node()) {
      selectedNodeId = null;
      selectedEdgeKey = null;

      model.set("selected_tag", {});
      model.save_changes();

      scheduleDraw();
    }
  });

  // --- COLONNA DESTRA: PANNELLO DI CONTROLLO ---
  const panel = mainContainer.append("div")
    .style("width", "320px")
    .style("height", "420px")
    .style("box-sizing", "border-box")
    .style("padding", "12px")
    .style("background", "#ffffff")
    .style("border", "1px solid #cbd5e1")
    .style("border-radius", "8px")
    .style("display", "flex")
    .style("flex-direction", "column")
    .style("gap", "12px")
    .style("overflow-y", "auto");

  // Legenda Tag
  const legendSection = panel.append("div")
    .style("padding-bottom", "10px")
    .style("border-bottom", "1px solid #e2e8f0");

  legendSection.append("div")
    .style("font-weight", "bold")
    .style("font-size", "13px")
    .style("margin-bottom", "6px")
    .text("*>> Tag Globali Attivi");

  const legendContainer = legendSection.append("div")
    .style("display", "flex")
    .style("flex-wrap", "wrap")
    .style("gap", "6px");

  // Sezione Dettagli Chunk
  const detailSection = panel.append("div")
    .style("display", "flex")
    .style("flex-direction", "column")
    .style("gap", "8px");

  detailSection.append("div")
    .style("font-weight", "bold")
    .style("font-size", "13px")
    .text(" °> Chunk Selezionato");

  const detailContent = detailSection.append("div")
    .style("font-size", "12px")
    .style("color", "#64748b");

  const detailHeader = detailContent.append("div").style("font-weight", "bold").style("color", "#0f172a").style("margin-bottom", "4px");
  const detailText = detailContent.append("div")
    .style("max-height", "90px")
    .style("overflow-y", "auto")
    .style("padding", "6px 8px")
    .style("background", "#f1f5f9")
    .style("border-radius", "4px")
    .style("font-size", "11px")
    .style("margin-bottom", "8px");

  detailContent.append("div").style("font-weight", "600").style("font-size", "11px").style("margin-bottom", "4px").text("Tag Associati:");
  const currentTagsContainer = detailContent.append("div")
    .style("display", "flex")
    .style("flex-wrap", "wrap")
    .style("gap", "4px")
    .style("margin-bottom", "8px");

  const addTagBox = detailContent.append("div")
    .style("display", "flex")
    .style("gap", "4px")
    .style("margin-bottom", "10px");

  const inputTag = addTagBox.append("input")
    .attr("type", "text")
    .attr("placeholder", "Nuovo Tag...")
    .style("flex", "1")
    .style("padding", "4px 6px")
    .style("border", "1px solid #cbd5e1")
    .style("border-radius", "4px")
    .style("font-size", "11px");

  const submitBtn = addTagBox.append("button")
    .text("+ Aggiungi")
    .style("padding", "4px 8px")
    .style("background", "#2563eb")
    .style("color", "#ffffff")
    .style("border", "none")
    .style("border-radius", "4px")
    .style("cursor", "pointer")
    .style("font-size", "11px");

  // Sezione Controllo Classe Arco / Distanza
  const edgeSection = panel.append("div")
    .style("display", "flex")
    .style("flex-direction", "column")
    .style("gap", "6px")
    .style("padding-top", "8px")
    .style("border-top", "1px solid #e2e8f0");

  edgeSection.append("div")
    .style("font-weight", "bold")
    .style("font-size", "13px")
    .text("<--> Modifica Classe Distanza Arco");

  const edgeContent = edgeSection.append("div")
    .style("font-size", "11px")
    .style("color", "#64748b");

  const edgeHeader = edgeContent.append("div").style("font-weight", "bold").style("color", "#0f172a").style("margin-bottom", "4px");
  const edgeButtonsContainer = edgeContent.append("div")
    .style("display", "flex")
    .style("flex-direction", "column")
    .style("gap", "4px");

  function submitTag() {
    const val = inputTag.property("value").trim();
    if (!val || !selectedNodeId) return;

    const graph = model.get("graph_data");
    const nodeData = graph && graph.nodes ? graph.nodes.find(n => str(n.id) === str(selectedNodeId)) : null;
    const userTags = nodeData ? (nodeData.user_tags || nodeData.tags || []) : [];

    if (!userTags.includes(val)) {
      model.set("tag_action", {
        action: "add",
        chunk_id: selectedNodeId,
        tag: val,
        timestamp: Date.now()
      });
      model.save_changes();
      inputTag.property("value", "");
    }
  }

  inputTag.on("keydown", (event) => { if (event.key === "Enter") submitTag(); });
  submitBtn.on("click", submitTag);

  function updateLegend() {
    const globalTags = model.get("global_tags") || {};
    legendContainer.html("");

    const tagNames = Object.keys(globalTags);
    if (tagNames.length === 0) {
      legendContainer.append("span").style("color", "#94a3b8").style("font-size", "11px").text("Nessun tag assegnato.");
      return;
    }

    tagNames.forEach(tagName => {
      const info = globalTags[tagName];
      legendContainer.append("span")
        .style("display", "inline-flex")
        .style("align-items", "center")
        .style("padding", "2px 8px")
        .style("border-radius", "12px")
        .style("font-size", "11px")
        .style("color", "#ffffff")
        .style("font-weight", "500")
        .style("background", info.color || "#6366f1")
        .text(`${tagName} (${info.count})`);
    });
  }

  function getNodeFill(d, globalTags) {
    const userTags = d.user_tags || d.tags || [];
    if (userTags.length === 0) return "#94a3b8";

    if (userTags.length === 1) {
      const tag = userTags[0];
      return (globalTags[tag] && globalTags[tag].color) ? globalTags[tag].color : "#6366f1";
    }

    const gradId = `grad-${str(d.id).replace(/[^a-zA-Z0-9_-]/g, '_')}`;
    defs.select(`#${gradId}`).remove();

    const grad = defs.append("linearGradient")
      .attr("id", gradId)
      .attr("x1", "0%").attr("y1", "0%")
      .attr("x2", "100%").attr("y2", "100%");

    const count = userTags.length;
    userTags.forEach((t, i) => {
      const color = (globalTags[t] && globalTags[t].color) ? globalTags[t].color : "#6366f1";
      const startPercent = (i / count) * 100;
      const endPercent = ((i + 1) / count) * 100;

      grad.append("stop").attr("offset", `${startPercent}%`).attr("stop-color", color);
      grad.append("stop").attr("offset", `${endPercent}%`).attr("stop-color", color);
    });

    return `url(#${gradId})`;
  }

  function renderEdgePanel(sourceId, targetId, currentClass) {
    edgeButtonsContainer.html("");
    if (!sourceId || !targetId) {
      edgeHeader.text("Clicca un arco per cambiarne la classe.");
      return;
    }

    edgeHeader.text(`Arco: ${sourceId} ↔ ${targetId}`);

    EDGE_CLASSES.forEach(cls => {
      // Evidenzia con bordo nero SOLO se l'arco ha una classe esplicita assegnata
      const isSelected = Boolean(currentClass) && currentClass.toUpperCase() === cls.name;

      const btn = edgeButtonsContainer.append("button")
        .text(cls.label)
        .style("padding", "4px 8px")
        .style("border", isSelected ? "2px solid #000000" : "1px solid #cbd5e1")
        .style("border-radius", "4px")
        .style("background", cls.color)
        .style("color", "#ffffff")
        .style("font-weight", "bold")
        .style("font-size", "10px")
        .style("cursor", "pointer")
        .style("text-align", "left");

      btn.on("click", (event) => {
        event.stopPropagation();
        model.set("pairwise_class_edit", {
          chunk_1: sourceId,
          chunk_2: targetId,
          distance_class: cls.name,
          timestamp: Date.now()
        });
        model.save_changes();
        selectedEdgeKey = getEdgeKey(sourceId, targetId);
        renderEdgePanel(sourceId, targetId, cls.name);
        scheduleDraw();
      });
    });
  }

  function renderDetailPanel() {
    const graph = model.get("graph_data");
    const globalTags = model.get("global_tags") || {};

    if (!selectedNodeId || !graph || !graph.nodes) {
      detailHeader.text("");
      detailText.text("Clicca un nodo nel grafo per vederne il testo e gestirne i Tag.");
      currentTagsContainer.html("");
      addTagBox.style("display", "none");
      return;
    }

    const nodeData = graph.nodes.find(n => str(n.id) === str(selectedNodeId));
    if (!nodeData) {
      detailHeader.text("");
      detailText.text("Nodo non trovato.");
      currentTagsContainer.html("");
      addTagBox.style("display", "none");
      return;
    }

    addTagBox.style("display", "flex");
    if (str(nodeData.id).startsWith("predq_")) {
      detailHeader.text(`*Q* Query Pre-determinata: ${nodeData.id}`).style("color", "#b45309");
    } else {
      detailHeader.text(`ID ${nodeData.id}`).style("color", "#0f172a");
    }
    detailText.text(nodeData.text || "Nessun testo associato.");

    currentTagsContainer.html("");
    const userTags = nodeData.user_tags || nodeData.tags || [];

    if (userTags.length === 0) {
      currentTagsContainer.append("span").style("font-size", "11px").style("color", "#94a3b8").text("Nessun tag.");
    } else {
      userTags.forEach(t => {
        const tagColor = (globalTags[t] && globalTags[t].color) ? globalTags[t].color : "#6366f1";
        const tagPill = currentTagsContainer.append("span")
          .style("display", "inline-flex")
          .style("align-items", "center")
          .style("gap", "4px")
          .style("padding", "2px 6px")
          .style("border-radius", "4px")
          .style("background", tagColor)
          .style("color", "#ffffff")
          .style("font-size", "11px");

        tagPill.append("span").text(t);
        tagPill.append("span")
          .style("cursor", "pointer")
          .style("font-weight", "bold")
          .style("margin-left", "2px")
          .text("✕")
          .on("click", (event) => {
            event.stopPropagation();
            model.set("tag_action", {
              action: "remove",
              chunk_id: nodeData.id,
              tag: t,
              timestamp: Date.now()
            });
            model.save_changes();
          });
      });
    }
  }

  function draw() {
    if (currentSimulation) {
      currentSimulation.stop();
    }

    g.selectAll("*").remove();
    defs.selectAll("*").remove();

    const graph = model.get("graph_data");
    const globalTags = model.get("global_tags") || {};

    updateLegend();
    renderDetailPanel();

    if (!selectedEdgeKey) {
      renderEdgePanel(null, null, null);
    }

    if (!graph || !graph.nodes || graph.nodes.length === 0) return;

    const posMap = {};
    if (currentNodes) {
      currentNodes.forEach(n => {
        posMap[str(n.id)] = { x: n.x, y: n.y, fx: n.fx, fy: n.fy };
      });
    }

    const nodes = graph.nodes.map(d => {
      const existing = posMap[str(d.id)];
      if (existing) {
        return { ...d, x: existing.x, y: existing.y, fx: existing.fx, fy: existing.fy };
      }
      return { ...d };
    });

    currentNodes = nodes;
    const links = graph.links ? graph.links.map(d => ({ ...d, isVirtual: false })) : [];

    // --- ARCHI SOTTO-SOGLIA (TRATTEGGIATI, NULL CLASS) PER NODO SELEZIONATO ---
    if (selectedNodeId) {
      const connectedTargets = new Set();
      links.forEach(l => {
        const s = str(typeof l.source === 'object' ? l.source.id : l.source);
        const t = str(typeof l.target === 'object' ? l.target.id : l.target);
        if (s === str(selectedNodeId)) connectedTargets.add(t);
        if (t === str(selectedNodeId)) connectedTargets.add(s);
      });

      nodes.forEach(n => {
        const nId = str(n.id);
        if (nId !== str(selectedNodeId) && !connectedTargets.has(nId)) {
          links.push({
            source: selectedNodeId,
            target: n.id,
            distance_class: null, // Nessuna classe preselezionata
            isVirtual: true
          });
        }
      });
    }

    const BASE_DISTANCE = graph.base_distance || 120;

    const simulation = d3.forceSimulation(nodes)
      .force("link", d3.forceLink(links)
        .id(d => d.id)
        .distance(BASE_DISTANCE)
      )
      .force("charge", d3.forceManyBody().strength(-180))
      .force("center", d3.forceCenter(width / 2, height / 2));

    currentSimulation = simulation;

    // --- RENDERING ARCHI ---
    const linkGroup = g.append("g")
      .selectAll("g")
      .data(links)
      .enter().append("g");

    // Hit-Area invisibile (12px) per facilitare il click
    const linkHitArea = linkGroup.append("line")
      .attr("stroke", "transparent")
      .attr("stroke-width", 12)
      .style("cursor", "pointer");

    // Linea visibile dell'arco
    const linkVisible = linkGroup.append("line")
      .attr("stroke", d => {
        const srcId = typeof d.source === 'object' ? d.source.id : d.source;
        const tgtId = typeof d.target === 'object' ? d.target.id : d.target;
        const key = getEdgeKey(srcId, tgtId);

        if (key === selectedEdgeKey) return "#0f172a";
        return d.isVirtual ? "#cbd5e1" : (EDGE_CLASS_COLORS[(d.distance_class || "INVARIATI").toUpperCase()] || "#94a3b8");
      })
      .attr("stroke-width", d => {
        const srcId = typeof d.source === 'object' ? d.source.id : d.source;
        const tgtId = typeof d.target === 'object' ? d.target.id : d.target;
        const key = getEdgeKey(srcId, tgtId);

        if (key === selectedEdgeKey) return 5.0;
        return d.isVirtual ? 1.5 : 2.5;
      })
      .attr("stroke-dasharray", d => d.isVirtual ? "4,4" : "none")
      .style("cursor", "pointer");

    // Gestore Click sull'Arco
    function handleEdgeClick(event, d) {
      event.stopPropagation();
      const srcId = typeof d.source === 'object' ? d.source.id : d.source;
      const tgtId = typeof d.target === 'object' ? d.target.id : d.target;

      selectedEdgeKey = getEdgeKey(srcId, tgtId);

      const initialClass = d.isVirtual ? d.distance_class : (d.distance_class || "INVARIATI");
      renderEdgePanel(srcId, tgtId, initialClass);
      scheduleDraw();
    }

    linkHitArea.on("click", handleEdgeClick);
    linkVisible.on("click", handleEdgeClick);

    // --- RENDERING NODI ---
    const node = g.append("g")
      .selectAll("circle")
      .data(nodes)
      .enter().append("circle")
      .attr("r", d => str(d.id).startsWith("predq_") ? 17 : 13) // Raggio maggiorato per Query
      .attr("fill", d => getNodeFill(d, globalTags))
      .attr("stroke", d => {
        if (str(d.id) === str(selectedNodeId)) return "#000000";
        if (str(d.id).startsWith("predq_")) return "#d97706"; // Bordo ambra/oro per Query
        return "#ffffff";
      })
      .attr("stroke-width", d => {
        if (str(d.id) === str(selectedNodeId)) return 4;
        if (str(d.id).startsWith("predq_")) return 3.5;
        return 2;
      })
      .style("cursor", "grab");

    const label = g.append("g")
      .selectAll("text")
      .data(nodes)
      .enter().append("text")
      .text(d => str(d.id).startsWith("predq_") ? `*Q* ${d.id}` : d.id)
      .attr("font-size", "11px")
      .attr("font-weight", d => str(d.id).startsWith("predq_") ? "bold" : "normal")
      .attr("dx", d => str(d.id).startsWith("predq_") ? 20 : 16)
      .attr("dy", 4)
      .attr("fill", d => str(d.id).startsWith("predq_") ? "#b45309" : "#1e293b");

    simulation.on("end", () => {
      nodes.forEach(n => {
        n.fx = n.x;
        n.fy = n.y;
      });
    });

    const drag = d3.drag()
      .container(g.node())
      .on("start", (event, d) => {
        nodes.forEach(n => {
          n.fx = n.x;
          n.fy = n.y;
        });
      })
      .on("drag", (event, d) => {
        d.fx = event.x;
        d.fy = event.y;
        d.x = event.x;
        d.y = event.y;
        updatePositions();
      })
      .on("end", (event, d) => {
        d.fx = event.x;
        d.fy = event.y;
        d.x = event.x;
        d.y = event.y;
        updatePositions();
      });

    node.call(drag);

    // Evento Click sul Nodo
    node.on("click", (event, d) => {
      event.stopPropagation();
      if (str(selectedNodeId) !== str(d.id)) {
        inputTag.property("value", "");
      }

      selectedNodeId = d.id;
      scheduleDraw();

      model.set("selected_tag", { id: d.id, text: d.text || "" });
      model.save_changes();
    });

    function updatePositions() {
      linkHitArea
        .attr("x1", d => d.source.x)
        .attr("y1", d => d.source.y)
        .attr("x2", d => d.target.x)
        .attr("y2", d => d.target.y);

      linkVisible
        .attr("x1", d => d.source.x)
        .attr("y1", d => d.source.y)
        .attr("x2", d => d.target.x)
        .attr("y2", d => d.target.y);

      node.attr("cx", d => d.x).attr("cy", d => d.y);
      label.attr("x", d => d.x).attr("y", d => d.y);
    }

    simulation.on("tick", updatePositions);
    updatePositions();
  }

  function scheduleDraw() {
    if (!renderPending) {
      renderPending = true;
      requestAnimationFrame(() => {
        renderPending = false;
        draw();
      });
    }
  }

  model.on("change:graph_data", scheduleDraw);
  model.on("change:global_tags", scheduleDraw);

  draw();

  return () => {
    if (currentSimulation) {
      currentSimulation.stop();
    }
    model.off("change:graph_data", scheduleDraw);
    model.off("change:global_tags", scheduleDraw);
  };
}

